# Causal Inference When You Cannot Randomise

Notebook for **Chapter 8** of the *ds* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/ds/ch08.html

Tip: change a number, predict the output, then run the cell again.

## 8.1 Potential Outcomes, DAGs and Confounding

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
sig = lambda x: 1/(1+np.exp(-x))
rng = np.random.default_rng(0); N = 20000
eng = rng.normal(0, 1, N)
t = (rng.random(N) < sig(-0.3 - 1.2*eng)).astype(int)       # low eng -> more offers
y = (rng.random(N) < sig(-0.5 + 1.0*eng + 0.4*t)).astype(int)
print(y[t==1].mean() - y[t==0].mean())                       # naive

## 8.2 Adjustment and Propensity Scores

In [ ]:
ps = LogisticRegression(C=1e6).fit(eng.reshape(-1,1), t).predict_proba(eng.reshape(-1,1))[:,1]
w = t/ps + (1-t)/(1-ps)
ipw = np.sum(w*t*y)/np.sum(w*t) - np.sum(w*(1-t)*y)/np.sum(w*(1-t))

## 8.3 Difference-in-Differences

In [ ]:
m = df.groupby(['treat','post']).y.mean().unstack()
did = (m.loc[1,1]-m.loc[1,0]) - (m.loc[0,1]-m.loc[0,0])

## 8.4 Synthetic Control

In [ ]:
from scipy.optimize import minimize
res = minimize(lambda w: np.sum((treated_y[:20] - donors[:20] @ w)**2),
               np.ones(J)/J, bounds=[(0,1)]*J,
               constraints={'type':'eq','fun': lambda w: w.sum()-1}, method='SLSQP')

## 8.5 Regression Discontinuity

## 8.6 Instrumental Variables

In [ ]:
ols    = np.polyfit(d, yv, 1)[0]
first  = np.polyfit(z, d, 1)[0]          # effect of SMS on auto-pay
reduced= np.polyfit(z, yv, 1)[0]         # effect of SMS on repayment
wald   = reduced / first

## 8.7 Uplift Modelling and Heterogeneous Effects

In [ ]:
m1 = GradientBoostingClassifier(random_state=0, max_depth=2, n_estimators=100).fit(X[tr&(t==1)], y[tr&(t==1)])
m0 = GradientBoostingClassifier(random_state=0, max_depth=2, n_estimators=100).fit(X[tr&(t==0)], y[tr&(t==0)])
uplift = m1.predict_proba(X[te])[:,1] - m0.predict_proba(X[te])[:,1]

## 8.8 Meta-Learners and Double Machine Learning in Depth

Section 8.7 introduced uplift with one T-learner on randomised data. Real life is harder in two ways. First, the treatment is often NOT randomised, so the model must also handle confounding. Second, you need to know which learner to trust. Here we put four estimators on one dataset where we know the true individual effect, and score each by how close it gets.

In [ ]:
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
rng = np.random.default_rng(8)
n = 20000
x1, x2 = rng.normal(size=n), rng.normal(size=n)
e = 1/(1+np.exp(-(0.8*x1)))                 # confounded treatment: x1 drives T
t = (rng.random(n) < e).astype(int)
tau = 1.0 + 1.5*x2                          # true effect depends on x2 only
y = 2*x1 + x2 + tau*t + rng.normal(size=n)
X = np.c_[x1, x2]
print("true ATE      ", round(tau.mean(), 3))
print("naive diff    ", round(y[t==1].mean() - y[t==0].mean(), 3))
def gb(): return GradientBoostingRegressor(n_estimators=150, max_depth=3, random_state=0)
# S-learner
s = gb().fit(np.c_[X, t], y)
tau_s = s.predict(np.c_[X, np.ones(n)]) - s.predict(np.c_[X, np.zeros(n)])
# T-learner
m1 = gb().fit(X[t==1], y[t==1]); m0 = gb().fit(X[t==0], y[t==0])
tau_t = m1.predict(X) - m0.predict(X)
# X-learner
d1 = y[t==1] - m0.predict(X[t==1]); d0 = m1.predict(X[t==0]) - y[t==0]
g1 = gb().fit(X[t==1], d1); g0 = gb().fit(X[t==0], d0)
tau_x = e*g0.predict(X) + (1-e)*g1.predict(X)   # true propensity used for weights
# Double ML (partialling out, 2-fold cross-fitting)
idx = rng.permutation(n); A, B = idx[:n//2], idx[n//2:]
ry = np.zeros(n); rt = np.zeros(n)
for tr, te in [(A, B), (B, A)]:
    ry[te] = y[te] - gb().fit(X[tr], y[tr]).predict(X[te])
    rt[te] = t[te] - gb().fit(X[tr], t[tr]).predict(X[te])
theta = (rt*ry).sum() / (rt*rt).sum()
# CATE by DML: regress ry on rt*[1, x2]
Z = np.c_[rt, rt*x2]
coef = np.linalg.lstsq(Z, ry, rcond=None)[0]
tau_d = coef[0] + coef[1]*x2
print(f"{'method':10s}{'mean':>8s}{'RMSE':>8s}{'corr':>8s}")
for name, v in [("S", tau_s), ("T", tau_t), ("X", tau_x), ("DML", tau_d)]:
    print(f"{name:10s}{v.mean():8.3f}{np.sqrt(((v-tau)**2).mean()):8.3f}{np.corrcoef(v, tau)[0,1]:8.3f}")
print("DML scalar theta", round(theta, 3), " DML slope on x2", round(coef[1], 3))

true ATE       0.999
naive diff     2.287
method        mean    RMSE    corr
S            0.969   0.250   0.988
T            0.976   0.280   0.983
X            0.969   0.171   0.994
DML          0.965   0.039   1.000
DML scalar theta 0.961  DML slope on x2 1.481


## 8.9 Staggered Rollouts: Why Two-Way Fixed Effects Can Fail

Section 8.3 compared one treated group with one control group at one launch date. Companies rarely do that. Stores get the new checkout in waves, cities get a feature month by month. The habit is to run one regression with a unit fixed effect, a time fixed effect and a 0/1 "treated now" dummy, called two-way fixed effects (TWFE). With one launch date it equals DiD. With several launch dates and effects that change over time, it can be badly wrong, even wrong in sign.

In [ ]:
import numpy as np
rng = np.random.default_rng(1)
U, T = 300, 12
cohort = np.repeat([4, 8, 99], 100)             # first treated period; 99 = never treated
ufe = rng.normal(0, 1, U); tfe = 0.3*np.arange(T)
def effect(c, t):
    if t < c: return 0.0
    return 0.5*(t - c + 1) if c == 4 else 0.5   # early cohort: grows; late cohort: flat 0.5
Y = np.zeros((U, T)); E = np.zeros((U, T)); D = np.zeros((U, T))
for u in range(U):
    for t in range(T):
        E[u, t] = effect(cohort[u], t); D[u, t] = t >= cohort[u]
        Y[u, t] = ufe[u] + tfe[t] + E[u, t] + rng.normal(0, 0.5)
print("true average effect over all treated cells:", round(E[D == 1].mean(), 3))
uu, tt = np.meshgrid(np.arange(U), np.arange(T), indexing="ij")
Uo = (uu.ravel()[:, None] == np.arange(1, U)[None, :]).astype(float)
To = (tt.ravel()[:, None] == np.arange(1, T)[None, :]).astype(float)
Z = np.c_[D.ravel(), np.ones(U*T), Uo, To]
b = np.linalg.lstsq(Z, Y.ravel(), rcond=None)[0]
print("TWFE estimate:", round(b[0], 3))
nv = cohort == 99; ests = []
for c in (4, 8):
    g = cohort == c
    for t in range(c, T):
        ests.append((Y[g, t] - Y[g, c-1]).mean() - (Y[nv, t] - Y[nv, c-1]).mean())
print("cohort-by-cohort DiD vs never-treated, averaged:", round(np.mean(ests), 3))

true average effect over all treated cells: 1.667
TWFE estimate: 0.883
cohort-by-cohort DiD vs never-treated, averaged: 1.703


## 8.10 One Treated Unit Over Time, and How Wrong Could We Be?

In [ ]:
import numpy as np
rng = np.random.default_rng(5)
T, T0 = 100, 70                                   # launch at day 70
common = np.cumsum(rng.normal(0, 1, T)) + 0.05*np.arange(T)
c1 = common + rng.normal(0, 0.8, T) + 10
c2 = 0.8*common + rng.normal(0, 0.8, T) + 20
c3 = 1.2*common + rng.normal(0, 0.8, T) + 5
treated = 1.0*common + rng.normal(0, 0.8, T) + 15
treated[T0:] += 3.0                               # true lift of 3 per day after launch
C = np.c_[np.ones(T), c1, c2, c3]
beta = np.linalg.lstsq(C[:T0], treated[:T0], rcond=None)[0]
resid = treated[:T0] - C[:T0] @ beta
sd = resid.std(ddof=4)
cf = C[T0:] @ beta
gap = treated[T0:] - cf
print("pre-period fit sd:", round(sd, 3))
print("average daily lift:", round(gap.mean(), 3))
# placebo: pretend launch at day 50 using only days 0..69
beta_p = np.linalg.lstsq(C[:50], treated[:50], rcond=None)[0]
gap_p = treated[50:70] - C[50:70] @ beta_p
print("placebo lift (days 50-69, no launch):", round(gap_p.mean(), 3))
# rough interval: sd of the mean of gap errors, inflated for autocorrelation by 2
se = 2*sd/np.sqrt(T-T0)
print("approx 95% interval:", round(gap.mean()-1.96*se, 2), "to", round(gap.mean()+1.96*se, 2))

pre-period fit sd: 0.874
average daily lift: 2.82
placebo lift (days 50-69, no launch): 0.02
approx 95% interval: 2.19 to 3.45


In [ ]:
import numpy as np
def evalue(rr):
    rr = 1/rr if rr < 1 else rr
    return rr + np.sqrt(rr*(rr-1))
print("E-value for RR 1.8:", round(evalue(1.8), 3))
print("E-value for CI limit 1.3:", round(evalue(1.3), 3))
print("E-value for RR 0.5:", round(evalue(0.5), 3))

E-value for RR 1.8: 3.0
E-value for CI limit 1.3: 1.924
E-value for RR 0.5: 3.414


## Chapter recap

## Practice set

**1. (easy)** Treated customers ordered at 40%, controls at 30%, but treated customers were twice as engaged on average. Can you call the 10-point gap the effect? **Answer:** No. Engagement likely confounds; adjust for it or randomise.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/ds/ch08.html